# ts_detrend_diff

> 对应代码：`EconometricsCode/code_in_notes/Chap.29/ts_detrend_diff.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.29`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.29")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们读入季度 GDP 数据，声明时间变量，并取对数得到 ln_gdp：对数化使序列的指数增长近似线性，差分后即可解释为增长率。

In [ ]:
%%stata
use ../datasets/quarterlyGDP.dta, clear
tsset time
gen t = _n
gen ln_gdp = log(qgdp)
label variable ln_gdp "log(GDP)"

首先对 ln_gdp 做含趋势项的增广 Dickey-Fuller（ADF）检验。原假设 H₀: ρ = 1（存在单位根）；备择假设为围绕线性趋势的平稳过程。trend 选项在检验回归中加入时间趋势，lags(5) 加入 5 阶滞后差分项以吸收残差中可能的自相关，regress 则要求打印出检验回归本身。结果通常无法拒绝单位根，说明对数 GDP 不是趋势平稳的。

In [ ]:
%%stata
// Unit root test
dfuller ln_gdp, trend regress lags(5)

对序列做一阶差分得到（近似）季度增长率 r1，再分别用含趋势和仅含漂移项的设定做 ADF 检验。差分后单位根通常被消除：含 trend 的检验对应原序列为带漂移的随机游走，drift 选项则明确允许差分方程中存在非零漂移（即 GDP 的平均增长）。

In [ ]:
%%stata
// difference
gen r1 = D.ln_gdp
dfuller r1, trend regress lags(5)
dfuller r1, drift regress lags(5)

季度数据往往存在季节性，因此我们改用与上年同季相比的差分 r2 = ln_gdp − ln_gdp(−4)，即同比增长率，它能同时消除单位根与确定性的季节波动。对 r2 同样做含趋势与含漂移的 ADF 检验加以确认。

In [ ]:
%%stata
// quarter difference
gen r2 = ln_gdp - L4.ln_gdp
dfuller r2, trend regress lags(5)
dfuller r2, drift regress lags(5)

最后只对 r1 回归一个常数项，检验差分后序列的均值（即平均增长率/漂移项）是否显著异于零。显著的正漂移意味着原始随机游走带漂移项，对应 ln_gdp 的近似线性增长趋势。

In [ ]:
%%stata
// test drift
reg r1